In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm import tqdm

sys.path.insert(0, '../../molearn/src/')
from molearn.data import PDBData
from molearn.models.foldingnet import AutoEncoder
from molearn.analysis import MolearnAnalysis
from molearn.analysis.plot import *
from molearn.analysis.path import oversample
from molearn.trainers.trainer import Trainer

import torch

In [ ]:
fig_dir = Path('analysis_bb_foldingnet_basic')
if fig_dir.exists() is False:
    fig_dir.mkdir(parents=True, exist_ok=True)

atoms = ["N", "CA", "C", "O", 'CB']

Load training data (open and closed state simulation) and test data (closed -> open) transition into separate PDBData classes.

In [ ]:
data_closed = PDBData(
    filename=["./data/MurD_closed.pdb"],
    atoms=atoms,
    fix_terminal=True,
)
_ = data_closed.prepare_dataset()

In [ ]:
data_open = PDBData(
    filename=["./data/MurD_open.pdb"],
    atoms=atoms,
    fix_terminal=True,
)
_ = data_open.prepare_dataset()

In [ ]:
data_test = PDBData(
    filename=["./data/MurD_closed_apo.pdb"],
    atoms=atoms,
    fix_terminal=True,
)
_ = data_test.prepare_dataset()

Load trained model checkpoint. Initialize MolearnAnalysis class and link to datasets and model.

In [ ]:
checkpoint_file = next((Path('foldingnet_checkpoints')).glob('checkpoint_converged.ckpt'))
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

checkpoint = torch.load(checkpoint_file, map_location=device, weights_only=False)
net = AutoEncoder(**checkpoint['network_kwargs'])
net.load_state_dict(checkpoint['model_state_dict'])
if torch.cuda.is_available():
    net.to(device)

In [ ]:
MA = MolearnAnalysis()
MA.set_network(net)
MA.batch_size = 16
MA.processes = 32

MA.set_dataset(data=data_open, key="train_open")
MA.set_dataset(data=data_closed, key="train_closed")
MA.set_dataset(data=data_test, key="test_trans")

Calculate and plot the reconstruction RMSD for datasets.

In [ ]:
rmsd_summary = {}
for key in ["train_open", "train_closed", "test_trans"]:
    errors = MA.get_error(key, align=True)
    rmsd_summary[key] = errors
    print(f"{key}: mean {errors.mean():.3f} Å ± {errors.std():.3f} Å (n={len(errors)})")

rmsd_plot_data = [
    ("train_open", "test_trans", "Open vs Transition", "train", "test"),
    ("train_closed", "test_trans", "Closed vs Transition", "train", "test"),
]

plot_rmsd_hist(
    MA,
    plot_data=rmsd_plot_data,
    fname=fig_dir / "rmsd_violin.png",
    dpi=150,
 )
print(f"Saved violin plot to {fig_dir / 'rmsd_violin.png'}")

Calculate and plot the bondlengths for datasets.

In [ ]:
bondlength_plot_data = [
    ("train_open", "Train Open", "#1f77b4"),
    ("train_closed", "Train Closed", "#ff7f0e"),
    ("test_trans", "Test Transition", "#2ca02c"),
]

plot_bondlength_hist(
    MA,
    plot_data=bondlength_plot_data,
    bins=60,
    wkdir=fig_dir,
    dpi=150,
)
print(f"Bond-length histograms saved under {fig_dir}")

Calculate the DOPE surface as a grid on latent space.

In [ ]:
if "grid" not in MA._encoded:
    grid_key = MA.setup_grid(samples=75, bounds_from=["train_open", "train_closed", "test_trans"])
    print(f"Latent grid '{grid_key}' initialised with {MA.n_samples} samples per axis.")
else:
    grid_key = "grid"
    print("Re-using previously initialised latent grid.")

dope_surface, xvals, yvals = MA.scan_dope(refine=True)
surface_clip = np.percentile(dope_surface, 95)

dope_plot_data = [
    ("train_open", "Train Open", "#FDBFCA", "scatter"),
    ("train_closed", "Train Closed", "#AFC2DC", "scatter"),
    ("test_trans", "Test Transition", "#7FB069", "scatter"),
]

plot_dope_surface(
    MA,
    refine=True,
    truncate_at=surface_clip,
    plot_data=dope_plot_data,
    cmap="viridis",
    fname=fig_dir / "dope_surface_refined.png",
    bbox_inches="tight",
)
print(
    f"Saved refined DOPE surface to {fig_dir / 'dope_surface_refined.png'} "
    f"(min={dope_surface.min():.1f}, max={dope_surface.max():.1f})",
)